# Aplicação de Wishlist com Django e SQLite
Uma aplicação para gerenciar uma wishlist com banco de dados local e funcionalidades de organização.

## 1. Configuração do Ambiente
Instalação e importação das bibliotecas necessárias

In [ ]:
# Instalação das dependências necessárias
!pip install django
!pip install pandas
!pip install openpyxl
!pip install pillow

In [ ]:
# Importação das bibliotecas
import django
from django.db import models
import pandas as pd
import sqlite3
import json
from datetime import datetime

## 2. Criação do Banco de Dados Local
Configuração inicial do SQLite e Django

In [ ]:
# Configuração do Django
import os
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'wishlist.settings')
django.setup()

# Criar conexão com SQLite
conn = sqlite3.connect('wishlist.db')
cursor = conn.cursor()

## 3. Definição do Modelo de Dados
Criação do modelo para os itens da wishlist

In [ ]:
# Definição do modelo Django
class WishlistItem(models.Model):
    nome = models.CharField(max_length=200)
    descricao = models.TextField()
    preco = models.DecimalField(max_digits=10, decimal_places=2)
    link = models.URLField()
    data_adicao = models.DateTimeField(auto_now_add=True)
    status = models.CharField(max_length=50)
    
    def __str__(self):
        return self.nome

## 4. Criação de Rotas para a API
Implementação das funções CRUD básicas

In [ ]:
def adicionar_item(nome, descricao, preco, link, status="Pendente"):
    cursor.execute("""
    INSERT INTO wishlist_items (nome, descricao, preco, link, status, data_adicao)
    VALUES (?, ?, ?, ?, ?, ?)
    """, (nome, descricao, preco, link, status, datetime.now()))
    conn.commit()

def listar_items():
    return pd.read_sql_query("SELECT * FROM wishlist_items", conn)

def atualizar_item(id, **kwargs):
    updates = ", ".join([f"{k} = ?" for k in kwargs.keys()])
    query = f"UPDATE wishlist_items SET {updates} WHERE id = ?"
    cursor.execute(query, list(kwargs.values()) + [id])
    conn.commit()

def deletar_item(id):
    cursor.execute("DELETE FROM wishlist_items WHERE id = ?", (id,))
    conn.commit()

## 5. Funções de Organização e Filtros
Implementação das funções de organização

In [ ]:
def ordenar_por_preco():
    return pd.read_sql_query("SELECT * FROM wishlist_items ORDER BY preco", conn)

def ordenar_por_nome():
    return pd.read_sql_query("SELECT * FROM wishlist_items ORDER BY nome", conn)

def filtrar_por_status(status):
    return pd.read_sql_query("SELECT * FROM wishlist_items WHERE status = ?", 
                            conn, params=[status])

## 6. Exportação de Dados
Funções para exportar os dados em diferentes formatos

In [ ]:
def exportar_para_excel(filename="wishlist.xlsx"):
    df = listar_items()
    df.to_excel(filename, index=False)

def exportar_para_csv(filename="wishlist.csv"):
    df = listar_items()
    df.to_csv(filename, index=False)

def exportar_para_json(filename="wishlist.json"):
    df = listar_items()
    df.to_json(filename, orient="records")

## 7. Geração de Lista no Formato Kanban
Função para gerar visualização Kanban dos items

In [ ]:
def gerar_kanban():
    df = listar_items()
    kanban = {
        "Pendente": df[df["status"] == "Pendente"],
        "Em Análise": df[df["status"] == "Em Análise"],
        "Comprado": df[df["status"] == "Comprado"]
    }
    
    for status, items in kanban.items():
        print(f"\n=== {status} ===")
        for _, item in items.iterrows():
            print(f"\n- {item['nome']}")
            print(f"  Preço: R$ {item['preco']}")
            print(f"  Link: {item['link']}")